# Solution 14 — A shipping expert and its eval case

The desk from notebook 14, plus a shipping expert. "Parcel questions go to shipping" is steering, so it
is one more line in `RULES` and the new tool's docstring. The shipping policy lives in the expert's own
instructions. Everything else (R1–R6) is unchanged. It all runs keyless and ends in `assert`s.

In [ ]:
from learningdantic import setup

MODEL = setup()

## The desk, with a shipping expert

Two cells copied from notebook 14. The new parts are `shipping_expert`, the `ask_shipping` tool and one
line in `RULES`.

In [ ]:
from dataclasses import dataclass, field

from pydantic_ai import Agent, FunctionToolset, RunContext
from pydantic_ai.models.test import TestModel


@dataclass
class Ticket:
    customer: str
    role: str = "customer"  # "customer" or "staff"
    audit_log: list[str] = field(default_factory=list)


ORDERS = {1042: "1 blue mug, delivered on Monday", 2077: "2 white mugs, in transit"}

billing_expert = Agent(
    TestModel(custom_output_text="A teammate reviews every refund request within 5 days."),
    name="billing_expert",
    instructions=(
        "You are Acme's billing expert. Refund policy: a teammate reviews every refund request "
        "within 5 days, and damaged mugs are replaced or refunded after that review. "
        "Never promise a refund. Answer in one sentence."
    ),
)

shipping_expert = Agent(
    TestModel(custom_output_text="Parcels take up to 7 days; after that we send a new one."),
    name="shipping_expert",
    instructions=(
        "You are Acme's shipping expert. Parcels take up to 7 days. After 7 days, Acme sends "
        "a new parcel for free. Answer in one sentence."
    ),
)

desk_tools = FunctionToolset()


@desk_tools.tool_plain
def lookup_order(order_id: int) -> str:
    """Look up one order by its number."""
    return ORDERS.get(order_id, f"There is no order {order_id}.")


@desk_tools.tool
async def ask_billing(ctx: RunContext, question: str) -> str:
    """Ask the billing expert about refunds, charges and invoices."""
    answer = await billing_expert.run(question, usage=ctx.usage, usage_limits=ctx.usage_limits)
    return answer.output


@desk_tools.tool
async def ask_shipping(ctx: RunContext, question: str) -> str:
    """Ask the shipping expert about late, lost or missing parcels."""
    answer = await shipping_expert.run(question, usage=ctx.usage, usage_limits=ctx.usage_limits)
    return answer.output


@desk_tools.tool_plain(requires_approval=True)  # R5
def issue_refund(order_id: int) -> str:
    """Refund an order. Staff only."""
    return f"Refunded order {order_id}."

In [ ]:
from typing import Literal

from pydantic import BaseModel
from pydantic_ai import DeferredToolRequests, ModelRetry, UsageLimits
from pydantic_ai.capabilities import Hooks

RULES = (  # R1
    "You are the support desk for Acme, a shop that sells mugs. "
    "Look up the order before you answer about it. "
    "For any refund or charge question, ask billing and pass its answer on. "
    "For a late, lost or missing parcel, ask shipping and pass its answer on. "
    "End every message with: Need more help? Reply HELP."
)
PROMISE_WORDS = ["guarantee", "full refund"]


class Reply(BaseModel):  # R2
    category: Literal["billing", "shipping", "other"]
    message: str


audit = Hooks()


@audit.on.before_tool_execute  # R4
def log_call(ctx, call, tool_def, args):
    ctx.deps.audit_log.append(call.tool_name)
    return args


desk = Agent(
    TestModel(),  # a stand-in; the live cell swaps in your MODEL
    name="desk",
    deps_type=Ticket,
    instructions=RULES,  # R1
    output_type=[Reply, DeferredToolRequests],  # R2 (DeferredToolRequests lets R5 pause a run)
    toolsets=[  # R3
        desk_tools.filtered(lambda ctx, td: td.name != "issue_refund" or ctx.deps.role == "staff")
    ],
    capabilities=[audit],  # R4
)


@desk.output_validator  # R2
def no_refund_promise(output: Reply) -> Reply:
    found = [word for word in PROMISE_WORDS if word in output.message.lower()]
    if found:
        raise ModelRetry(
            f"Never promise a refund (you wrote {found}). Say a teammate reviews it within 5 days."
        )
    return output


LIMITS = UsageLimits(request_limit=6)  # R6


async def handle(text: str, ticket: Ticket):
    """The desk's one front door: every ticket runs with the same limits."""
    return await desk.run(text, deps=ticket, usage_limits=LIMITS)

## Part 1 — the new specialist

A customer must be shown `ask_shipping`, and an answer from the expert must reach the reply. The
scripted desk below routes by keywords and builds its reply from what the `ask_…` tools returned.
`ToolReturnPart.tool_name` says which tool a result came from.

In [ ]:
from pydantic_ai.messages import ModelResponse, ToolCallPart, ToolReturnPart
from pydantic_ai.models.function import FunctionModel


def offered(test_model):
    """Names of the tools the model was shown on its last request (from 09)."""
    return [tool.name for tool in test_model.last_model_request_parameters.function_tools]


def fake_desk(messages, info):
    """Scripted desk: look up order 1042, send money to billing and parcels to shipping, then reply."""
    text = messages[0].parts[-1].content.lower()
    money = "money" in text or "refund" in text
    parcel = "parcel" in text
    if len(messages) == 1:  # the first request: call the tools
        calls = [ToolCallPart("lookup_order", {"order_id": 1042})]
        if money:
            calls.append(ToolCallPart("ask_billing", {"question": text}))
        if parcel:
            calls.append(ToolCallPart("ask_shipping", {"question": text}))
        return ModelResponse(parts=calls)
    experts = [
        part.content
        for part in messages[-1].parts
        if isinstance(part, ToolReturnPart) and part.tool_name.startswith("ask_")
    ]
    if money:
        category = "billing"
    elif parcel:
        category = "shipping"
    else:
        category = "other"
    reply = {"category": category, "message": " ".join(experts + ["Need more help? Reply HELP."])}
    return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, reply)])


model = TestModel(call_tools=[])
with desk.override(model=model):
    await handle("Where is my parcel?", Ticket("Ada"))
print("customer sees:", offered(model))

dan = Ticket("Dan")
with desk.override(model=FunctionModel(fake_desk)):
    result = await handle("My parcel for order 2077 is two weeks late.", dan)
print(result.output)
print("audit log:", sorted(dan.audit_log), "| requests:", result.usage.requests)

assert "ask_shipping" in offered(model)
assert "issue_refund" not in offered(model)  # R3 still holds
assert result.output.category == "shipping"
assert "after that we send a new one" in result.output.message  # the expert's answer
assert result.usage.requests == 3  # desk 2 + shipping 1, on one bill
assert sorted(dan.audit_log) == ["ask_shipping", "lookup_order"]  # R4 logs the new tool too

## Part 2 — the eval case

The same `desk_evals` as notebook 14, plus `late_parcel`. The cases about refunds and status still pass,
so adding shipping broke nothing.

In [ ]:
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import Contains, IsInstance, ToolCorrectness


async def answer(text: str) -> str:
    """The eval task: one customer ticket in, the desk's message out."""
    result = await handle(text, Ticket("eval"))
    return result.output.message


desk_evals = Dataset(
    name="desk",
    cases=[
        Case(
            name="refund",
            inputs="My mug from order 1042 arrived cracked. I want my money back.",
            evaluators=[ToolCorrectness(expected_tools=["lookup_order", "ask_billing"])],
        ),
        Case(
            name="status",
            inputs="Has order 1042 arrived yet?",
            evaluators=[ToolCorrectness(expected_tools=["lookup_order"])],
        ),
        Case(
            name="late_parcel",
            inputs="My parcel for order 2077 is two weeks late.",
            evaluators=[ToolCorrectness(expected_tools=["lookup_order", "ask_shipping"])],
        ),
    ],
    evaluators=[IsInstance("str"), Contains("Reply HELP")],
)
with desk.override(model=FunctionModel(fake_desk)):
    report = await desk_evals.evaluate(answer, name="scripted desk", progress=False)
report.print(include_durations=False)

late = [case for case in report.cases if case.name == "late_parcel"][0]
assert late.assertions["ToolCorrectness"].value, late.assertions["ToolCorrectness"].reason
assert report.averages().assertions == 1.0
print("✅ the shipping expert is wired in, logged, budgeted and covered by an eval case")